# Karachi AQI — Exploratory Data Analysis
Loads 1-year feature history from Hopsworks and analyzes AQI patterns, pollutant drivers, seasonal trends, and correlations.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()
%matplotlib inline
sns.set_theme(style='whitegrid')
print('Libraries loaded')

In [ ]:
import hopsworks
from src.config import HOPSWORKS_API_KEY, HOPSWORKS_PROJECT, FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION
project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY, project=HOPSWORKS_PROJECT)
fs = project.get_feature_store()
fg = fs.get_feature_group(FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
df = fg.read()
df = df.sort_values('datetime').reset_index(drop=True)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# AQI over time — full year
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['datetime'], df['aqi'], linewidth=0.6, alpha=0.8, color='steelblue')
ax.axhline(150, color='orange', linestyle='--', label='Unhealthy threshold')
ax.axhline(200, color='red', linestyle='--', label='Very Unhealthy')
ax.set_title('Karachi AQI — Full Year')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition
from statsmodels.tsa.seasonal import seasonal_decompose
daily_aqi = df.set_index('datetime')['aqi'].resample('D').mean().dropna()
decomp = seasonal_decompose(daily_aqi, model='additive', period=30)
decomp.plot()
plt.suptitle('AQI Seasonal Decomposition (30-day period)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# AQI by hour of day
if df['datetime'].dt.tz is not None:
    df_local = df.copy()
    df_local['hour'] = df_local['datetime'].dt.hour
else:
    df_local = df.copy()
    df_local['hour'] = df_local['datetime'].dt.hour
hourly_avg = df_local.groupby('hour')['aqi'].mean()
fig, ax = plt.subplots(figsize=(10, 4))
hourly_avg.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Average AQI by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Mean AQI')
plt.tight_layout()
plt.show()

In [ ]:
# AQI by day of week
df_local['dow'] = df_local['datetime'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg = df_local.groupby('dow')['aqi'].mean().reindex(dow_order)
fig, ax = plt.subplots(figsize=(8, 4))
dow_avg.plot(kind='bar', color='teal', ax=ax)
ax.set_title('Average AQI by Day of Week')
ax.set_xlabel('')
ax.set_ylabel('Mean AQI')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap: features vs AQI
from src.config import TABULAR_FEATURES
corr_cols = [c for c in TABULAR_FEATURES if c in df.columns] + ['aqi']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Pollutant sub-index breakdown (which pollutant drives AQI most often?)
from src.aqi import _sub_index, PM25_BREAKPOINTS, PM10_BREAKPOINTS, O3_PPB_BREAKPOINTS, NO2_PPB_BREAKPOINTS, SO2_PPB_BREAKPOINTS, CO_PPM_BREAKPOINTS
_O3_CONV = 1.9957; _NO2_CONV = 1.9125; _SO2_CONV = 2.6196; _CO_CONV = 1145.6
df['si_pm25'] = df['pm2_5'].apply(lambda x: _sub_index(x, PM25_BREAKPOINTS))
df['si_pm10'] = df['pm10'].apply(lambda x: _sub_index(x, PM10_BREAKPOINTS))
df['si_o3'] = df['ozone'].apply(lambda x: _sub_index(x / _O3_CONV, O3_PPB_BREAKPOINTS))
df['si_no2'] = df['nitrogen_dioxide'].apply(lambda x: _sub_index(x / _NO2_CONV, NO2_PPB_BREAKPOINTS))
df['si_so2'] = df['sulphur_dioxide'].apply(lambda x: _sub_index(x / _SO2_CONV, SO2_PPB_BREAKPOINTS))
df['si_co'] = df['carbon_monoxide'].apply(lambda x: _sub_index(x / _CO_CONV, CO_PPM_BREAKPOINTS))
si_cols = ['si_pm25','si_pm10','si_o3','si_no2','si_so2','si_co']
dominant = df[si_cols].idxmax(axis=1).value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
dominant.plot(kind='pie', autopct='%1.1f%%', ax=ax)
ax.set_title('Which Pollutant Drives AQI Most Often?')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
print('Dominant sub-index counts:', dominant.to_dict())

In [ ]:
# Extreme AQI events (> 300)
extremes = df[df['aqi'] > 300][['datetime','aqi','pm2_5','pm10','ozone','temperature_2m','wind_speed_10m']]
print(f'Extreme events (AQI > 300): {len(extremes)}')
extremes.head(20)